# 01 · Land the next SAP extract
Stands in for an SAP extractor (AecorSoft / SAP Datasphere) dropping ODP delta
files into the landing volume. batch_id = auto picks the next batch.

In [0]:
import os, sys

# This notebook lives in src/notebooks; the package lives in src/.
# Add src/ to Python's search path so `import sap_extract_simulator` works.
sys.path.append(os.path.abspath(".."))

from sap_extract_simulator import next_batch_id, write_batch, marker_path

In [0]:
print("cwd:", os.getcwd())
print("src contents:", os.listdir(os.path.abspath("..")))
print("package contents:", os.listdir(os.path.abspath("../sap_extract_simulator")))

In [0]:
dbutils.widgets.text("landing_root", "/Volumes/workspace/sap_bronze/landing/sap")
dbutils.widgets.text("batch_id", "auto")

landing_root = dbutils.widgets.get("landing_root")
batch_param = dbutils.widgets.get("batch_id").strip().lower()
batch_id = next_batch_id(landing_root) if batch_param in ("", "auto") else int(batch_param)
print(f"Writing SAP extract batch {batch_id} to {landing_root}")

In [0]:
paths = write_batch(batch_id, landing_root)

for table, path in paths.items():
    with open(path) as fh:
        rows = sum(1 for _ in fh) - 1          # minus the header line
    print(f"{table:5s} {rows:5d} rows  ->  {path}")

print("marker:", marker_path(landing_root, batch_id), os.path.exists(marker_path(landing_root, batch_id)))

In [0]:
from pyspark.sql import functions as F

kna1 = spark.read.option("header", True).csv(f"{landing_root}/KNA1/extract_{batch_id:04d}.csv")
display(kna1.limit(10))

# How many creates / updates / deletes in this batch?
display(kna1.groupBy("ODQ_CHANGEMODE").count())

In [0]:
vbap = spark.read.option("header", True).csv(f"{landing_root}/VBAP/extract_{batch_id:04d}.csv")

bad_items = vbap.where(
    F.col("MATNR").isNull() | (F.col("KWMENG").cast("int") <= 0)
)
print("bad VBAP rows:", bad_items.count())
display(bad_items)

print("KNA1 rows with no key:", kna1.where(F.col("KUNNR").isNull()).count())